<a href="https://colab.research.google.com/github/ajsksbxdjd/model-training/blob/main/updated_with_unseen_data_colab_p15_p3_only.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hookworm Detector: P1.5 + P3.0 Only

This notebook trains and compares YOLO hookworm egg detectors using only the P1.5 and P3.0 datasets.

## 1. Mount Drive and Set Paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
from google.colab import userdata
import os

FYP = Path('/content/drive/MyDrive/FYP')
FYP.mkdir(parents=True, exist_ok=True)

WORK = Path('/content/work')
WORK.mkdir(parents=True, exist_ok=True)
DATA_ROOT = WORK / 'datasets'
OUT = DATA_ROOT / 'hookworm_p15_p3_only'
RUNS_DIR = FYP / 'runs_p15_p3_only'

kaggle_username = userdata.get('KAGGLE_USERNAME')
kaggle_key = userdata.get('KAGGLE_KEY')
if kaggle_username and kaggle_key:
    os.environ['KAGGLE_USERNAME'] = kaggle_username
    os.environ['KAGGLE_KEY'] = kaggle_key
    print('Kaggle credentials loaded from Colab Secrets.')
else:
    print('Kaggle credentials were not found in Colab Secrets.')
    print('Add KAGGLE_USERNAME and KAGGLE_KEY in the Colab Secrets panel before running the download cell.')

print('Work directory:', WORK)
print('Output dataset:', OUT)
print('Runs directory:', RUNS_DIR)

Kaggle credentials loaded from Colab Secrets.
Work directory: /content/work
Output dataset: /content/work/datasets/hookworm_p15_p3_only
Runs directory: /content/drive/MyDrive/FYP/runs_p15_p3_only


## 2. Install and Import Libraries

In [3]:
!pip install ultralytics kagglehub tensorflow pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 79.8 MB/s eta 0:00:00


In [4]:
import io
import random
import shutil
import zipfile
from pathlib import Path
from collections import defaultdict

import kagglehub
import pandas as pd
import tensorflow as tf
from PIL import Image as PILImage
from ultralytics import YOLO

SEED = 42
random.seed(SEED)

# Keep some negative examples so the detector learns when NOT to predict hookworm.
# 1.5 means at most 1.5 negative images for every positive image within each source.
NEGATIVE_RATIO = 1.5

print('Ready')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ready


## 3. Download P3.0 and P1.5

In [5]:
print('Downloading P3.0...')
p3_path = Path(kagglehub.dataset_download('peterkward/ai4ntd-p3'))
print(f'  P3.0: {p3_path}')

print('Downloading P1.5...')
p15_path = Path(kagglehub.dataset_download('peterkward/ai4ntd-p1-5'))
print(f'  P1.5: {p15_path}')

Using Colab cache for faster access to the 'ai4ntd-p3' dataset.
  P3.0: /kaggle/input/ai4ntd-p3
Using Colab cache for faster access to the 'ai4ntd-p1-5' dataset.
  P1.5: /kaggle/input/ai4ntd-p1-5


## 4. Build P1.5 + P3.0 Samples

In [6]:
def yolo_line(cls, cx, cy, w, h):
    cx = min(max(cx, 0.0), 1.0)
    cy = min(max(cy, 0.0), 1.0)
    w = min(max(w, 0.0), 1.0)
    h = min(max(h, 0.0), 1.0)
    return f'{cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}'


def make_sample(item, yolo_lines, stem, is_bytes, source, fixed_split=None):
    return {
        'item': item,
        'yolo_lines': yolo_lines,
        'stem': stem,
        'is_bytes': is_bytes,
        'source': source,
        'fixed_split': fixed_split,
        'positive': len(yolo_lines) > 0,
    }


def cap_negatives(samples, ratio=NEGATIVE_RATIO):
    positives = [s for s in samples if s['positive']]
    negatives = [s for s in samples if not s['positive']]
    random.shuffle(negatives)
    if not positives:
        return negatives[:200]
    max_negatives = int(len(positives) * ratio)
    return positives + negatives[:max_negatives]


all_samples = []

# P3.0: hookworm class 2 -> class 0. Non-hookworm images become negative examples.
print('Processing P3.0...')
p3_img_dir = p3_path / 'group1/images'
p3_lbl_dir = p3_path / 'group1/labels'
p3_source_samples = []

for img_file in sorted(p3_img_dir.iterdir()):
    if img_file.suffix.lower() not in {'.jpg', '.jpeg', '.png'}:
        continue
    lbl_file = p3_lbl_dir / f'{img_file.stem}.txt'
    lines = []
    if lbl_file.exists():
        lines = [line.strip() for line in lbl_file.read_text().splitlines() if line.strip()]
    hookworm_lines = [line for line in lines if line.split()[0] == '2']
    yolo_lines = ['0' + line[1:] for line in hookworm_lines]
    p3_source_samples.append(make_sample(img_file, yolo_lines, f'p3_{img_file.stem}', False, 'p3'))

p3_source_samples = cap_negatives(p3_source_samples)
all_samples.extend(p3_source_samples)
print(f'  P3.0 kept: {len(p3_source_samples)} images | positives: {sum(s["positive"] for s in p3_source_samples)} | negatives: {sum(not s["positive"] for s in p3_source_samples)}')

# P1.5: hookworm label 3 -> class 0. Keep original train/eval/test split.
print('Processing P1.5...')

def extract_p15_tfrecord(record_path, prefix, fixed_split):
    samples = []
    dataset = tf.data.TFRecordDataset(str(record_path))
    for i, raw in enumerate(dataset):
        example = tf.train.Example()
        example.ParseFromString(raw.numpy())
        feat = example.features.feature
        img_bytes = feat['image/encoded'].bytes_list.value[0]
        xmins = list(feat['image/object/bbox/xmin'].float_list.value)
        xmaxs = list(feat['image/object/bbox/xmax'].float_list.value)
        ymins = list(feat['image/object/bbox/ymin'].float_list.value)
        ymaxs = list(feat['image/object/bbox/ymax'].float_list.value)
        labels = list(feat['image/object/class/label'].int64_list.value)

        yolo_lines = []
        for xmin, xmax, ymin, ymax, lbl in zip(xmins, xmaxs, ymins, ymaxs, labels):
            if int(lbl) == 3:
                yolo_lines.append(yolo_line(0, (xmin + xmax) / 2, (ymin + ymax) / 2, xmax - xmin, ymax - ymin))
        samples.append(make_sample(img_bytes, yolo_lines, f'{prefix}_{i:06d}', True, prefix, fixed_split))
    return cap_negatives(samples)

p15_train_samples = extract_p15_tfrecord(p15_path / 'train.record', 'p15_train', 'train')
p15_val_samples = extract_p15_tfrecord(p15_path / 'eval.record', 'p15_eval', 'val')
p15_test_samples = extract_p15_tfrecord(p15_path / 'test.record', 'p15_test', 'test')
all_samples.extend(p15_train_samples + p15_val_samples + p15_test_samples)
print(f'  P1.5 kept: train={len(p15_train_samples)}, val={len(p15_val_samples)}, test={len(p15_test_samples)}')

print(f'\nTotal P1.5 + P3.0 samples: {len(all_samples)}')
print(f'Total positives: {sum(s["positive"] for s in all_samples)}')
print(f'Total negatives: {sum(not s["positive"] for s in all_samples)}')

Processing P3.0...
  P3.0 kept: 140 images | positives: 56 | negatives: 84
Processing P1.5...
  P1.5 kept: train=3767, val=1125, test=590

Total P1.5 + P3.0 samples: 5622
Total positives: 2249
Total negatives: 3373


## 5. Create Train, Validation, and Unseen Test Splits

In [7]:
# Build train/val/test. P1.5 keeps its original split; P3.0 is split source-by-source.
if OUT.exists():
    shutil.rmtree(OUT)
for split in ['train', 'val', 'test']:
    (OUT / split / 'images').mkdir(parents=True, exist_ok=True)
    (OUT / split / 'labels').mkdir(parents=True, exist_ok=True)

splits = {'train': [], 'val': [], 'test': []}
free_by_source = defaultdict(list)

for sample_item in all_samples:
    fixed = sample_item['fixed_split']
    if fixed:
        splits[fixed].append(sample_item)
    else:
        free_by_source[sample_item['source']].append(sample_item)

for source, source_samples in free_by_source.items():
    random.shuffle(source_samples)
    n = len(source_samples)
    n_test = max(1, int(n * 0.15)) if n >= 3 else 0
    n_val = max(1, int(n * 0.15)) if n >= 3 else 0
    splits['test'].extend(source_samples[:n_test])
    splits['val'].extend(source_samples[n_test:n_test + n_val])
    splits['train'].extend(source_samples[n_test + n_val:])


def write_samples(samples, split):
    out_img = OUT / split / 'images'
    out_lbl = OUT / split / 'labels'
    rows = []
    for s in samples:
        stem = s['stem']
        try:
            if s['is_bytes']:
                img_path = out_img / f'{stem}.jpg'
                PILImage.open(io.BytesIO(s['item'])).save(img_path)
            else:
                src = Path(s['item'])
                img_path = out_img / f'{stem}{src.suffix.lower()}'
                shutil.copy2(src, img_path)
            label_path = out_lbl / f'{stem}.txt'
            label_path.write_text('\n'.join(s['yolo_lines']), encoding='utf-8')
            rows.append({
                'split': split,
                'source': s['source'],
                'stem': stem,
                'image': str(img_path),
                'label': str(label_path),
                'positive': s['positive'],
                'boxes': len(s['yolo_lines']),
            })
        except Exception as exc:
            print(f'  Skip {stem}: {exc}')
    return rows

manifest_rows = []
for split, samples in splits.items():
    manifest_rows.extend(write_samples(samples, split))

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(OUT / 'manifest.csv', index=False)

yaml_content = f'''path: {OUT}
train: train/images
val: val/images
test: test/images

nc: 1
names: ['hookworm_egg']
'''
(OUT / 'dataset.yaml').write_text(yaml_content, encoding='utf-8')

summary = manifest.groupby(['split', 'source', 'positive']).size().rename('count').reset_index()
print(summary)
print(f'\nDataset YAML: {OUT / "dataset.yaml"}')
print(f'Manifest: {OUT / "manifest.csv"}')

    split     source  positive  count
0    test   p15_test     False    354
1    test   p15_test      True    236
2    test         p3     False     16
3    test         p3      True      5
4   train  p15_train     False   2260
5   train  p15_train      True   1507
6   train         p3     False     59
7   train         p3      True     39
8     val   p15_eval     False    675
9     val   p15_eval      True    450
10    val         p3     False      9
11    val         p3      True     12

Dataset YAML: /content/work/datasets/hookworm_p15_p3_only/dataset.yaml
Manifest: /content/work/datasets/hookworm_p15_p3_only/manifest.csv


## 6. Export the Unseen Test Set

In [8]:
# Export the unseen test set to Drive so you can download it and test manually later.
unseen_zip = FYP / 'hookworm_unseen_test_manual_p15_p3_only.zip'
if unseen_zip.exists():
    unseen_zip.unlink()

readme = f'''Unseen hookworm test set: P1.5 + P3.0 only

This zip contains images and YOLO-format labels from the held-out test split.
The models in this notebook do not train on these images.

Manual prediction example after training:
model = YOLO('{RUNS_DIR / 'hookworm_yolov8s_960' / 'weights' / 'best.pt'}')
model.predict(source='{OUT / 'test/images'}', conf=0.15, save=True)

Ground-truth labels are included in test/labels so you can inspect misses and false positives.
'''

with zipfile.ZipFile(unseen_zip, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for path in (OUT / 'test').rglob('*'):
        if path.is_file():
            z.write(path, arcname=str(path.relative_to(OUT)))
    z.writestr('README.txt', readme)
    z.write(OUT / 'dataset.yaml', arcname='dataset.yaml')
    z.write(OUT / 'manifest.csv', arcname='manifest.csv')

print(f'Created unseen manual test zip: {unseen_zip}')
print(f'Test images: {len(list((OUT / "test/images").iterdir()))}')

Created unseen manual test zip: /content/drive/MyDrive/FYP/hookworm_unseen_test_manual_p15_p3_only.zip
Test images: 611


## 7. Train and Compare YOLO Models

In [9]:
# Train and compare multiple models.
# yolov8n is fastest. yolov8s is usually stronger. Uncomment yolov8m if you have time/GPU memory.
MODEL_CONFIGS = [
    {'weights': 'yolov8n.pt', 'name': 'hookworm_yolov8n_960', 'imgsz': 960, 'batch': 16},
    {'weights': 'yolov8s.pt', 'name': 'hookworm_yolov8s_960', 'imgsz': 960, 'batch': 16},
    {'weights': 'yolov8m.pt', 'name': 'hookworm_yolov8m_960', 'imgsz': 960, 'batch': 8},
]

EPOCHS = 150
PATIENCE = 40

def metric_row(model_name, weights, split, metrics):
    return {
        'model': model_name,
        'weights': weights,
        'split': split,
        'precision': float(metrics.box.p[0]),
        'recall': float(metrics.box.r[0]),
        'map50': float(metrics.box.ap50[0]),
        'map50_95': float(metrics.box.ap[0]),
    }

comparison_rows = []
trained_models = []

for cfg in MODEL_CONFIGS:
    print(f'\n=== Training {cfg["name"]} from {cfg["weights"]} ===')
    model = YOLO(cfg['weights'])
    model.train(
        data=str(OUT / 'dataset.yaml'),
        epochs=EPOCHS,
        imgsz=cfg['imgsz'],
        batch=cfg['batch'],
        name=cfg['name'],
        project=str(RUNS_DIR),
        patience=PATIENCE,
        save=True,
        plots=True,
        device=0,
        close_mosaic=20,
        seed=SEED,
        deterministic=True,
    )

    best_path = RUNS_DIR / cfg['name'] / 'weights' / 'best.pt'
    trained_models.append({'name': cfg['name'], 'weights': cfg['weights'], 'best_path': str(best_path)})
    best_model = YOLO(str(best_path))

    print(f'Validating {cfg["name"]} on val split...')
    val_metrics = best_model.val(data=str(OUT / 'dataset.yaml'), split='val', imgsz=cfg['imgsz'])
    comparison_rows.append(metric_row(cfg['name'], cfg['weights'], 'val', val_metrics))

    print(f'Validating {cfg["name"]} on unseen test split...')
    test_metrics = best_model.val(data=str(OUT / 'dataset.yaml'), split='test', imgsz=cfg['imgsz'])
    comparison_rows.append(metric_row(cfg['name'], cfg['weights'], 'test', test_metrics))

comparison = pd.DataFrame(comparison_rows).sort_values(['split', 'map50_95'], ascending=[True, False])
comparison_path = FYP / 'model_comparison_p15_p3_only.csv'
comparison.to_csv(comparison_path, index=False)
print('\nModel comparison:')
display(comparison)
print(f'Saved: {comparison_path}')


=== Training hookworm_yolov8n_960 from yolov8n.pt ===
Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=20, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/work/datasets/hookworm_p15_p3_only/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=hookworm_

,model,weights,split,precision,recall,map50,map50_95
1,hookworm_yolov8n_960,yolov8n.pt,test,0.961985,0.963412,0.982054,0.921787
3,hookworm_yolov8s_960,yolov8s.pt,test,0.959010,0.956853,0.979867,0.917606
5,hookworm_yolov8m_960,yolov8m.pt,test,0.968316,0.961929,0.981487,0.909378
0,hookworm_yolov8n_960,yolov8n.pt,val,0.956935,0.959569,0.986914,0.919168
2,hookworm_yolov8s_960,yolov8s.pt,val,0.960405,0.952830,0.988151,0.915867
4,hookworm_yolov8m_960,yolov8m.pt,val,0.951096,0.969003,0.987737,0.915267


Saved: /content/drive/MyDrive/FYP/model_comparison_p15_p3_only.csv


## 8. Save Best Model Summary

In [10]:
# Pick the best model by unseen-test mAP50-95 and save a short summary.
test_rows = comparison[comparison['split'] == 'test'].copy()
best = test_rows.sort_values('map50_95', ascending=False).iloc[0]
best_info = next(item for item in trained_models if item['name'] == best['model'])

summary_text = f'''Best model by unseen-test mAP50-95: P1.5 + P3.0 only

model: {best['model']}
starting weights: {best['weights']}
best.pt: {best_info['best_path']}
precision: {best['precision']:.4f}
recall: {best['recall']:.4f}
mAP50: {best['map50']:.4f}
mAP50-95: {best['map50_95']:.4f}

Unseen manual test zip: {unseen_zip}
Comparison CSV: {comparison_path}
'''

summary_path = FYP / 'best_model_summary_p15_p3_only.txt'
summary_path.write_text(summary_text, encoding='utf-8')
print(summary_text)
print(f'Saved: {summary_path}')

Best model by unseen-test mAP50-95: P1.5 + P3.0 only

model: hookworm_yolov8n_960
starting weights: yolov8n.pt
best.pt: /content/drive/MyDrive/FYP/runs_p15_p3_only/hookworm_yolov8n_960/weights/best.pt
precision: 0.9620
recall: 0.9634
mAP50: 0.9821
mAP50-95: 0.9218

Unseen manual test zip: /content/drive/MyDrive/FYP/hookworm_unseen_test_manual_p15_p3_only.zip
Comparison CSV: /content/drive/MyDrive/FYP/model_comparison_p15_p3_only.csv

Saved: /content/drive/MyDrive/FYP/best_model_summary_p15_p3_only.txt


## 9. Package Training Runs

In [11]:
# Show the saved best model files.
import os
for p in RUNS_DIR.glob('**/weights/best.pt'):
    print(p, round(os.path.getsize(p) / 1e6, 1), 'MB')

/content/drive/MyDrive/FYP/runs_p15_p3_only/hookworm_yolov8n_960/weights/best.pt 6.3 MB
/content/drive/MyDrive/FYP/runs_p15_p3_only/hookworm_yolov8s_960/weights/best.pt 22.6 MB
/content/drive/MyDrive/FYP/runs_p15_p3_only/hookworm_yolov8m_960/weights/best.pt 52.1 MB


In [12]:
# Zip all training runs to Drive for easy download.
archive_base = FYP / 'all_runs_p15_p3_only'
archive_path = shutil.make_archive(str(archive_base), 'zip', str(RUNS_DIR))
print('Created:', archive_path)

Created: /content/drive/MyDrive/FYP/all_runs_p15_p3_only.zip
